# AdaGrad Simulations - Non-Convex

## Parameters:

In [0]:
from Solver import AdaGrad, NonlocalSolverAdaGrad
from sklearn.model_selection import ParameterGrid
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import jax 
import jax.numpy as jnp
import os

param_grid = {'lr': [0.1, 0.01]}
n_learning_rates = len(param_grid['lr'])
param_list = list(ParameterGrid(param_grid))

dL = lambda y: y * (y**2 - 1) # Derivative of the function to minimize
f = lambda x, y: 0.0 # Rhs of the ODE without the nonlocal part

# Crear carpeta para guardar figuras si no existe
figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)

## AdaGrad - Discrete

In [0]:
# --- AdaGrad | Guardar PNG por condición inicial separada ---

inits = [0.1, -0.1]

for theta_initial in inits:
    # Crear subplots para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_accumulated_gradients_result = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Iterar por learning rates
    for i, lr in enumerate(param_grid['lr']):

        # Filtrar parámetros con este LR
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs según LR (manteniendo tus valores)
        if lr == 0.1:
            epochs = 50
        elif lr == 0.01:
            epochs = 4000

        # Simulaciones AdaGrad
        for params in filtered_params:
            print(f'\nAdaGrad Configuration: {params}, theta_initial={theta_initial}')

            solver = AdaGrad(dL=dL, lr=lr, epochs=epochs)
            solver.solve(theta_initial=theta_initial)

            # Theta_k
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='lines',
                showlegend=False,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # G_k (acumulado)
            fig_accumulated_gradients_result.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.accumulated_gradients_result,
                mode='markers',
                marker=dict(size=3),
                legendgroup=f'LR={lr}',
                showlegend=False
            ), row=1, col=i+1)

    # Layouts para ESTA condición inicial
    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectories for AdaGrad — θ₀={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_theta.update_xaxes(title_text="k")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta_k")

    fig_accumulated_gradients_result.update_layout(
        title_text=f'Accumulated gradients convergence trajectories for AdaGrad — θ₀={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_accumulated_gradients_result.update_xaxes(title_text="k")
    fig_accumulated_gradients_result.update_yaxes(title_text="G_k")

    # Sufijo de archivo (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Guardar PNGs para ESTA condición inicial
    fig_theta.write_image(os.path.join(figures_dir, f"adagrad_theta_{suffix}.png"))
    fig_accumulated_gradients_result.write_image(os.path.join(figures_dir, f"adagrad_gradients_{suffix}.png"))

    print(f"Figuras guardadas (θ₀={theta_initial}) en la carpeta '{figures_dir}'")

## Nonlocal AdaGrad

In [0]:
# --- Nonlocal Continuous AdaGrad | Guardar PNG por condición inicial separada ---

inits = [0.1, -0.1]

for theta_initial in inits:
    # 1) Crear subplots para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_G = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectory — Nonlocal Continuous AdaGrad (θ₀={theta_initial})'
    )
    fig_G.update_layout(
        title_text=f'G over time — Nonlocal Continuous AdaGrad (θ₀={theta_initial})'
    )

    # 2) Bucle por learning rates
    for i, lr in enumerate(param_grid['lr']):
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs según LR (manteniendo tus valores)
        if lr == 0.1:
            epochs = 50
        elif lr == 0.01:
            epochs = 4000

        t = [1e-12, epochs * lr]

        # 3) Simulaciones por conjunto de parámetros
        for params in filtered_params:
            print(f'\nNonlocal Continuous AdaGrad Configuration: {params}, theta_initial={theta_initial}')

            solver = NonlocalSolverAdaGrad(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]), alpha=params['lr']
            )
            t_values, y_values = solver.solve()

            # Theta(t)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                showlegend=False,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # G(t) guardado en solver
            denominators = np.asarray(solver._last_G)  # columnas [t, G]
            fig_G.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='markers',
                marker=dict(size=3),
                showlegend=False,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

    # 4) Ejes y tamaños (para ESTA θ0)
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta(t)")
    fig_theta.update_layout(width=1500, height=600)

    fig_G.update_xaxes(title_text="t/alpha")
    fig_G.update_yaxes(title_text="G(t)")
    fig_G.update_layout(width=1500, height=600)

    # 5) Guardado con sufijo por condición inicial (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    fig_theta.write_image(os.path.join(figures_dir, f"nonlocal_adagrad_theta_ncvx_{suffix}.png"))
    fig_G.write_image(os.path.join(figures_dir, f"nonlocal_g_gradient_ncvx_{suffix}.png"))

    print(f"Figuras guardadas (θ₀={theta_initial}) en la carpeta '{figures_dir}'")

## Both Models together

In [0]:
# --- AdaGrad vs. Nonlocal AdaGrad | PNGs separados por condición inicial ---
inits = [0.1, -0.1]

for theta_initial in inits:
    # Crear figuras para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1,
        cols=2,  # una columna por learning rate
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    fig_gradients = make_subplots(
        rows=1,
        cols=2,  # una columna por learning rate
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Iterar por learning rates
    for i, lr in enumerate(param_grid['lr']):
        # Filtrar parámetros (mismo LR)
        filtered_params_adagrad  = [p for p in param_list if p['lr'] == lr]
        filtered_params_nonlocal = [p for p in param_list if p['lr'] == lr]

        # Epochs / span temporal
        if lr == 0.1:
            epochs = 50
        elif lr == 0.01:
            epochs = 4000

        t = [1e-12, epochs * lr]

        # -------- AdaGrad (discreto) --------
        for params in filtered_params_adagrad:
            print(f'\nAdaGrad Configuration: {params}, theta_initial={theta_initial}')

            solver = AdaGrad(dL=dL, lr=lr, epochs=epochs)
            solver.solve(theta_initial=theta_initial)

            # Theta_k
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color='red'),
                name='AdaGrad',
                legendgroup='AdaGrad',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # Gradientes acumulados G_k
            fig_gradients.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.accumulated_gradients_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color='red'),
                name='AdaGrad Gradients',
                legendgroup='AdaGrad Gradients',
                showlegend=(i == 0)
            ), row=1, col=i+1)

        # ---- AdaGrad continuo no local ----
        for params in filtered_params_nonlocal:
            print(f'\nNonlocal Continuous AdaGrad Configuration: {params}, theta_initial={theta_initial}')

            solver_nonlocal = NonlocalSolverAdaGrad(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]), alpha=params['lr']
            )
            t_values, y_values = solver_nonlocal.solve()

            # θ(t)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                line=dict(color='blue'),
                name='Nonlocal AdaGrad',
                legendgroup='Nonlocal AdaGrad',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # G(t)
            denominators = np.asarray(solver_nonlocal._last_G)  # columnas [t, G]
            fig_gradients.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='lines',
                line=dict(color='blue'),
                name='Nonlocal AdaGrad Gradients',
                legendgroup='Nonlocal AdaGrad Gradients',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # Layouts (para ESTA θ0)
    fig_theta.update_layout(
        title_text=f'Theta Convergence Trajectories for Nonlocal AdaGrad — θ₀={theta_initial}',
        width=1200, height=600
    )
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta Values")

    fig_gradients.update_layout(
        title_text=f'Gradients Trajectories for Nonlocal AdaGrad — θ₀={theta_initial}',
        width=1200, height=600
    )
    fig_gradients.update_xaxes(title_text="t/alpha")
    fig_gradients.update_yaxes(title_text="Gradients Values")

    # Sufijo de archivo (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Guardar PNGs para ESTA condición inicial
    fig_theta.write_image(os.path.join(figures_dir, f"adagrad_vs_nonlocal_theta_ncvx_{suffix}.png"))
    fig_gradients.write_image(os.path.join(figures_dir, f"adagrad_vs_nonlocal_g_ncvx_{suffix}.png"))

    print(f"Figuras guardadas (θ₀={theta_initial}) en la carpeta '{figures_dir}'")